# 04. Feature Engineering

**Objective:** Create leakage-safe quarterly historical features for 338 grid cells
and merge the six road-network aggregates. The modeling dataset covers
2020Q1–2025Q4; 2019 supplies the initial history. Run from `notebooks/` after notebook 03.

## 1. Load and prepare the datasets

In [22]:
import pandas as pd

quarterly_panel = pd.read_csv(
    "../data/processed/quarterly_panel.csv"
)

graph_features = pd.read_csv(
    "../data/processed/graph_features.csv"
)

saved_graph_cols = [
    "road_length_m",
    "major_length_m",
    "major_road_share",
    "intersection_count",
    "intersection_density",
    "mean_streets_per_node"
]

assert graph_features.shape == (338, 7)
assert graph_features["cell_id"].is_unique
assert set(graph_features.columns) == {"cell_id", *saved_graph_cols}
graph_features = graph_features[["cell_id"] + saved_graph_cols]

quarterly_panel["quarter"] = pd.PeriodIndex(
    quarterly_panel["quarter"],
    freq="Q"
)

quarterly_panel = quarterly_panel.sort_values(
    ["cell_id", "quarter"]
).reset_index(drop=True)

assert not quarterly_panel.duplicated(["cell_id", "quarter"]).any()
assert quarterly_panel["quarter"].nunique() == 28
assert quarterly_panel.groupby("cell_id").size().eq(28).all()


In [23]:
print("Graph features:", graph_features.shape)
print("Quarterly panel:", quarterly_panel.shape)
print(graph_features.columns.tolist())

Graph features: (338, 7)
Quarterly panel: (9464, 3)
['cell_id', 'road_length_m', 'major_length_m', 'major_road_share', 'intersection_count', 'intersection_density', 'mean_streets_per_node']


## 2. Historical feature engineering

Each cell's rows are ordered by quarter. Lag features use previous observations, and `shift(1)` excludes the target quarter before the four-quarter rolling mean is calculated. Drop the first four quarters, which lack a full year of history.

In [24]:
# Group collision history by grid cell
grouped = quarterly_panel.groupby("cell_id")["collisions"]

# Previous-quarter collisions
quarterly_panel["lag_1"] = grouped.shift(1)

# Same quarter in the previous year
quarterly_panel["lag_4"] = grouped.shift(4)

# Average collisions over the previous 4 quarters
quarterly_panel["rolling_mean_4"] = (
    quarterly_panel.groupby("cell_id")["collisions"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

# Target quarter (1, 2, 3, or 4)
quarterly_panel["quarter_of_year"] = (
    quarterly_panel["quarter"].dt.quarter
)

In [25]:
model_data = quarterly_panel.dropna(
    subset=["lag_1", "lag_4", "rolling_mean_4"]
).copy()

In [26]:
sample = model_data[
    (model_data["quarter"] == pd.Period("2020Q1")) &
    (model_data["lag_1"] > 0) &
    (model_data["lag_4"] > 0)
].head(1)

display(sample)


,cell_id,quarter,collisions,lag_1,lag_4,rolling_mean_4,quarter_of_year
480,17,2020Q1,11,26.0,30.0,29.25,1


## 3. Merge graph features

Retain all six road-network aggregates in the saved dataset. Models C and D use four of them: road length, major-road share, intersection density, and mean streets per node. Major-road length and intersection count remain available as the underlying aggregates.

In [27]:
# Merge datasets
model_data = model_data.merge(
    graph_features,
    on="cell_id",
    how="left",
    validate="many_to_one"
)

In [28]:
# Define the feature sets used for model training
historical_cols = [
    "lag_1",
    "lag_4",
    "rolling_mean_4",
    "quarter_of_year"
]

graph_cols = [
    "road_length_m",
    "major_road_share",
    "intersection_density",
    "mean_streets_per_node"
]


## 4. Verify and save

In [29]:
# Preserve cell-level chronological ordering and the original dataset schema
model_data = model_data[
    ["cell_id", "quarter", "collisions"]
    + historical_cols + saved_graph_cols
].sort_values(["cell_id", "quarter"]).reset_index(drop=True)

assert model_data.shape == (8112, 13)
assert model_data[historical_cols + saved_graph_cols].notna().all().all()
assert not model_data.duplicated(["cell_id", "quarter"]).any()
assert model_data["cell_id"].nunique() == 338
assert model_data.groupby("cell_id").size().eq(24).all()
assert model_data["quarter"].min() == pd.Period("2020Q1")
assert model_data["quarter"].max() == pd.Period("2025Q4")

model_data.to_csv(
    "../data/processed/model_data.csv",
    index=False
)

print("Modeling dataset:", model_data.shape)
print("Historical features:", len(historical_cols))
print("Graph features for training:", len(graph_cols))
print("Saved road-network features:", len(saved_graph_cols))


Modeling dataset: (8112, 13)
Historical features: 4
Graph features for training: 4
Saved road-network features: 6


## 5. Summary

The verified `model_data.csv` contains 8,112 observations across 338 cells and
24 quarters (2020Q1–2025Q4), with 13 columns and no duplicate cell-quarter keys or
missing required features. The four historical features use only earlier observations;
the rolling mean applies `shift(1)` before its four-quarter window.

All six road-network aggregates are saved. Notebook 05 trains Model B with four
historical features and Models C and D with those four plus the four selected
road-network features.